In [1]:
import pandas as pd
import numpy as np 


In [2]:
project_details_df = pd.read_csv("/Users/drishtant/Documents/Masters/CA/core market data/project_details.csv")
country_region_df = pd.read_excel("/Users/drishtant/Documents/Masters/CA/core market data/Country - Region Mapping.xlsx")

In [3]:
# =the actual key for merging depends on matching country names or regions)
#  we need to create a mapping based on country names to regions 
#or directly use HDI rankings as a proxy for development status.

# Example merge operation (this is conceptual and might require adjustments
#based on actual column names and matching logic)
project_details_df = project_details_df.merge(country_region_df[['Region', 'Country Status']], left_on='region_name',
                                              right_on='Region', how='left')



In [4]:
project_details_df

,Unnamed: 0,id,product_class,product_type,product_type_name,product_type_long_name,certificate_project_type,certificate_project_type_name,name,description,...,un_sd_goals,project_identifier,region_short_name,certificate_name,product_type_region,un_sd_goal_colors,project_method_type,project_method_type_name,Region,Country Status
0,0,40,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,Grid Connected Wind Power Project in Maharashtra,M/s Bhilwara Green Energy limited (BGEL) is th...,...,"[7, 8, 13]",3705,India,GS,118.0,"['#fbc412', '#a31c44', '#407f46']",18,Renewables,India,Developing
1,1,47,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,300 MW Wind Energy Project,The purpose of the project activity is to gene...,...,"[4, 7, 8, 13, 15]",7468,India,GS,118.0,"['#c5192d', '#fbc412', '#a31c44', '#407f46', '...",18,Renewables,India,Developing
2,2,87,carbon,gs,GS (VER),Gold Standard (VER),31,Energy efficiency,Promoting Improved Cooking Practices in Nigeri...,The project involves manufacturing and distrib...,...,"[1, 7, 13]",7312,NaN,GS,NaN,"['#e5233d', '#fbc412', '#407f46']",31,Energy efficiency,NaN,NaN
3,3,346,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,Kartaldagi Wind Power Plant,Efil Enerji √úretim Ticaret ve Sanayi A.S. (Ef...,...,"[6, 7, 8, 13]",4922,Turkey,GS,202.0,"['#27bfe6', '#fbc412', '#a31c44', '#407f46']",18,Renewables,Turkey,High Developed
4,4,430,carbon,gs,GS (VER),Gold Standard (VER),31,Energy efficiency,"VPA 175 ECOZOOM IMPROVED STOVE PROGRAMME, UGANDA",The VPA involves the distribution of fuel-effi...,...,"[3, 7, 13]",7345,Uganda,GS,206.0,"['#4ca146', '#fbc412', '#407f46']",31,Energy efficiency,Uganda,Under Developed
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,156,301,carbon,accu,ACCU,ACCU,7,Avoided deforestation,Osterley Downs Native Forest Protection Project,\t\nThis project protects the native forest fr...,...,[13],EOP100653,NSW,ACCU,30.0,['#407f46'],7,Avoided deforestation,Australia,High Developed
157,157,82,carbon,accu,ACCU,ACCU,6,HIR,Ashwood Native Forest Protection Project,This project protects the native forest from b...,...,[13],EOP100580,NSW,ACCU,30.0,['#407f46'],6,HIR,Australia,High Developed
158,158,81,carbon,accu,ACCU,ACCU,4,Landfill gas capture,Swanbank Landfill Gas Project,\t\nThis project transitioned from a revoked (...,...,[13],EOP100172,Qld,ACCU,29.0,['#407f46'],4,Landfill gas capture,Australia,High Developed
159,159,75,carbon,accu,ACCU,ACCU,6,HIR,Paroo River North Environmental Project,\t\nThis project establishes permanent native ...,...,"[2, 3, 4, 13, 14, 15]",ERF104646,Qld,ACCU,29.0,"['#dda73a', '#4ca146', '#c5192d', '#407f46', '...",6,HIR,Australia,High Developed


In [5]:
project_details_df.to_csv("/Users/drishtant/Documents/Masters/CA/core market data/Project_details_1.1.csv")


In [6]:
test = project_details_df[project_details_df['id']== '408']

In [7]:
test['Region'].unique()
# 

array([], dtype=object)

In [8]:
project_details_df['project_method_type_name'].unique()

array(['Renewables', 'Energy efficiency', 'Community-based', nan,
       'Re/afforestation', 'REDD+', 'IFM', 'HIR', 'Landfill gas capture',
       'Savanna burning', 'Savanna burning+', 'Avoided deforestation'],
      dtype=object)

In [9]:
project_details_df['Country Status'].unique()

array(['Developing', nan, 'High Developed', 'Under Developed',
       'Developed'], dtype=object)

In [10]:
country_factor = {
    'Developing': 1.4,  # Reflects increased reliance on manual labor.
    'High Developed': 0.7,  # Indicates higher automation and efficiency.
    'Under Developed': 1.7,  # Suggests substantial reliance on manual labor due to limited access to technology.
    'Developed': 1,  # A balance between automation and manual labor.
}


In [11]:
methodology_factor = {
    'Renewables': 1.2,  # Labor-intensive, especially during construction and maintenance phases.
    'Energy efficiency': 1.0,  # Requires specialized labor but less extensively than renewables.
    'Community-based': 1.3,  # Highly labor-intensive due to community engagement and manual processes.
    'Re/afforestation': 1.4,  # Labor-intensive activities such as planting and maintenance.
    'REDD+': 1.1,  # Involves community engagement and monitoring activities.
    'IFM': 1.1,  # Improved Forest Management requires significant labor for implementation and monitoring.
    'HIR': 1.0,  # Human-induced Regeneration may vary in labor intensity.
    'Landfill gas capture': 0.9,  # Technologically driven, moderate labor requirements.
    'Savanna burning': 1.2,  # Manual fire management practices.
    'Savanna burning+': 1.2,  # Similar to savanna burning with potentially additional activities.
    'Avoided deforestation': 1.1,  # Monitoring and enforcement, involving community participation.
}


In [12]:
import pandas as pd

# Assuming project_details_df is loaded with the relevant columns
# Define base job offer ranges
base_ranges = {'Micro Scale': (1, 10), 'Small Scale': (11, 50), 'Medium Scale': (51, 200), 'Large Scale': (200, 1500)}

# Function to calculate adjusted job offer range
def calculate_adjusted_range(row):
    scale_range = base_ranges.get(row['Scale'], (0, 0))
    adjustment = country_factor.get(row['Country Status'], 1) * methodology_factor.get(row['project_method_type_name'], 1)
    adjusted_min, adjusted_max = (scale_range[0] * adjustment, scale_range[1] * adjustment)
    return f"{int(adjusted_min)}-{int(adjusted_max)}"

# # Function to calculate adjusted job offered
# def calculate_adjusted_range(row):
    #as we have range 

# Apply the function to the dataframe
project_details_df['Adjusted Job Offer Range'] = project_details_df.apply(calculate_adjusted_range, axis=1)


In [13]:
def calculate_exact_job_count(row):
    # Define the midpoint of each range as the representative value
    representative_values = {
        'Micro Scale': 5,  # Midpoint of 1-10
        'Small Scale': 30,  # Midpoint of 11-50
        'Medium Scale': 125,  # Midpoint of 51-200
        'Large Scale': 850
        
        
        ,  # Midpoint of 201-1000
    }
    
    # Retrieve base representative value for the project's scale
    base_job_count = representative_values.get(row['Scale'], 0)
    
    # Adjust the base job count based on country status and project methodology
    country_adj = country_factor.get(row['Country Status'], 1)
    method_adj = methodology_factor.get(row['project_method_type_name'], 1)
    
    # Calculate the adjusted job count
    adjusted_job_count = int(base_job_count * country_adj * method_adj)
    
    return adjusted_job_count

# Apply the function to each row in the DataFrame
project_details_df['Estimated Job Count'] = project_details_df.apply(calculate_exact_job_count, axis=1)


In [14]:
project_details_df

,Unnamed: 0,id,product_class,product_type,product_type_name,product_type_long_name,certificate_project_type,certificate_project_type_name,name,description,...,region_short_name,certificate_name,product_type_region,un_sd_goal_colors,project_method_type,project_method_type_name,Region,Country Status,Adjusted Job Offer Range,Estimated Job Count
0,0,40,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,Grid Connected Wind Power Project in Maharashtra,M/s Bhilwara Green Energy limited (BGEL) is th...,...,India,GS,118.0,"['#fbc412', '#a31c44', '#407f46']",18,Renewables,India,Developing,336-2520,1428
1,1,47,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,300 MW Wind Energy Project,The purpose of the project activity is to gene...,...,India,GS,118.0,"['#c5192d', '#fbc412', '#a31c44', '#407f46', '...",18,Renewables,India,Developing,336-2520,1428
2,2,87,carbon,gs,GS (VER),Gold Standard (VER),31,Energy efficiency,Promoting Improved Cooking Practices in Nigeri...,The project involves manufacturing and distrib...,...,NaN,GS,NaN,"['#e5233d', '#fbc412', '#407f46']",31,Energy efficiency,NaN,NaN,200-1500,850
3,3,346,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,Kartaldagi Wind Power Plant,Efil Enerji √úretim Ticaret ve Sanayi A.S. (Ef...,...,Turkey,GS,202.0,"['#27bfe6', '#fbc412', '#a31c44', '#407f46']",18,Renewables,Turkey,High Developed,168-1260,714
4,4,430,carbon,gs,GS (VER),Gold Standard (VER),31,Energy efficiency,"VPA 175 ECOZOOM IMPROVED STOVE PROGRAMME, UGANDA",The VPA involves the distribution of fuel-effi...,...,Uganda,GS,206.0,"['#4ca146', '#fbc412', '#407f46']",31,Energy efficiency,Uganda,Under Developed,1-17,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,156,301,carbon,accu,ACCU,ACCU,7,Avoided deforestation,Osterley Downs Native Forest Protection Project,\t\nThis project protects the native forest fr...,...,NSW,ACCU,30.0,['#407f46'],7,Avoided deforestation,Australia,High Developed,154-1155,654
157,157,82,carbon,accu,ACCU,ACCU,6,HIR,Ashwood Native Forest Protection Project,This project protects the native forest from b...,...,NSW,ACCU,30.0,['#407f46'],6,HIR,Australia,High Developed,140-1050,595
158,158,81,carbon,accu,ACCU,ACCU,4,Landfill gas capture,Swanbank Landfill Gas Project,\t\nThis project transitioned from a revoked (...,...,Qld,ACCU,29.0,['#407f46'],4,Landfill gas capture,Australia,High Developed,32-126,78
159,159,75,carbon,accu,ACCU,ACCU,6,HIR,Paroo River North Environmental Project,\t\nThis project establishes permanent native ...,...,Qld,ACCU,29.0,"['#dda73a', '#4ca146', '#c5192d', '#407f46', '...",6,HIR,Australia,High Developed,7-35,21


In [15]:
test = project_details_df[project_details_df['id'] == 408]

In [16]:
project_details_df

,Unnamed: 0,id,product_class,product_type,product_type_name,product_type_long_name,certificate_project_type,certificate_project_type_name,name,description,...,region_short_name,certificate_name,product_type_region,un_sd_goal_colors,project_method_type,project_method_type_name,Region,Country Status,Adjusted Job Offer Range,Estimated Job Count
0,0,40,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,Grid Connected Wind Power Project in Maharashtra,M/s Bhilwara Green Energy limited (BGEL) is th...,...,India,GS,118.0,"['#fbc412', '#a31c44', '#407f46']",18,Renewables,India,Developing,336-2520,1428
1,1,47,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,300 MW Wind Energy Project,The purpose of the project activity is to gene...,...,India,GS,118.0,"['#c5192d', '#fbc412', '#a31c44', '#407f46', '...",18,Renewables,India,Developing,336-2520,1428
2,2,87,carbon,gs,GS (VER),Gold Standard (VER),31,Energy efficiency,Promoting Improved Cooking Practices in Nigeri...,The project involves manufacturing and distrib...,...,NaN,GS,NaN,"['#e5233d', '#fbc412', '#407f46']",31,Energy efficiency,NaN,NaN,200-1500,850
3,3,346,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,Kartaldagi Wind Power Plant,Efil Enerji √úretim Ticaret ve Sanayi A.S. (Ef...,...,Turkey,GS,202.0,"['#27bfe6', '#fbc412', '#a31c44', '#407f46']",18,Renewables,Turkey,High Developed,168-1260,714
4,4,430,carbon,gs,GS (VER),Gold Standard (VER),31,Energy efficiency,"VPA 175 ECOZOOM IMPROVED STOVE PROGRAMME, UGANDA",The VPA involves the distribution of fuel-effi...,...,Uganda,GS,206.0,"['#4ca146', '#fbc412', '#407f46']",31,Energy efficiency,Uganda,Under Developed,1-17,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,156,301,carbon,accu,ACCU,ACCU,7,Avoided deforestation,Osterley Downs Native Forest Protection Project,\t\nThis project protects the native forest fr...,...,NSW,ACCU,30.0,['#407f46'],7,Avoided deforestation,Australia,High Developed,154-1155,654
157,157,82,carbon,accu,ACCU,ACCU,6,HIR,Ashwood Native Forest Protection Project,This project protects the native forest from b...,...,NSW,ACCU,30.0,['#407f46'],6,HIR,Australia,High Developed,140-1050,595
158,158,81,carbon,accu,ACCU,ACCU,4,Landfill gas capture,Swanbank Landfill Gas Project,\t\nThis project transitioned from a revoked (...,...,Qld,ACCU,29.0,['#407f46'],4,Landfill gas capture,Australia,High Developed,32-126,78
159,159,75,carbon,accu,ACCU,ACCU,6,HIR,Paroo River North Environmental Project,\t\nThis project establishes permanent native ...,...,Qld,ACCU,29.0,"['#dda73a', '#4ca146', '#c5192d', '#407f46', '...",6,HIR,Australia,High Developed,7-35,21


In [17]:
project_details_df.to_csv('/Users/drishtant/Documents/Masters/CA/core market data/Project_details_1.2.csv')

In [18]:
data = project_details_df

In [19]:
data

,Unnamed: 0,id,product_class,product_type,product_type_name,product_type_long_name,certificate_project_type,certificate_project_type_name,name,description,...,region_short_name,certificate_name,product_type_region,un_sd_goal_colors,project_method_type,project_method_type_name,Region,Country Status,Adjusted Job Offer Range,Estimated Job Count
0,0,40,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,Grid Connected Wind Power Project in Maharashtra,M/s Bhilwara Green Energy limited (BGEL) is th...,...,India,GS,118.0,"['#fbc412', '#a31c44', '#407f46']",18,Renewables,India,Developing,336-2520,1428
1,1,47,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,300 MW Wind Energy Project,The purpose of the project activity is to gene...,...,India,GS,118.0,"['#c5192d', '#fbc412', '#a31c44', '#407f46', '...",18,Renewables,India,Developing,336-2520,1428
2,2,87,carbon,gs,GS (VER),Gold Standard (VER),31,Energy efficiency,Promoting Improved Cooking Practices in Nigeri...,The project involves manufacturing and distrib...,...,NaN,GS,NaN,"['#e5233d', '#fbc412', '#407f46']",31,Energy efficiency,NaN,NaN,200-1500,850
3,3,346,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,Kartaldagi Wind Power Plant,Efil Enerji √úretim Ticaret ve Sanayi A.S. (Ef...,...,Turkey,GS,202.0,"['#27bfe6', '#fbc412', '#a31c44', '#407f46']",18,Renewables,Turkey,High Developed,168-1260,714
4,4,430,carbon,gs,GS (VER),Gold Standard (VER),31,Energy efficiency,"VPA 175 ECOZOOM IMPROVED STOVE PROGRAMME, UGANDA",The VPA involves the distribution of fuel-effi...,...,Uganda,GS,206.0,"['#4ca146', '#fbc412', '#407f46']",31,Energy efficiency,Uganda,Under Developed,1-17,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,156,301,carbon,accu,ACCU,ACCU,7,Avoided deforestation,Osterley Downs Native Forest Protection Project,\t\nThis project protects the native forest fr...,...,NSW,ACCU,30.0,['#407f46'],7,Avoided deforestation,Australia,High Developed,154-1155,654
157,157,82,carbon,accu,ACCU,ACCU,6,HIR,Ashwood Native Forest Protection Project,This project protects the native forest from b...,...,NSW,ACCU,30.0,['#407f46'],6,HIR,Australia,High Developed,140-1050,595
158,158,81,carbon,accu,ACCU,ACCU,4,Landfill gas capture,Swanbank Landfill Gas Project,\t\nThis project transitioned from a revoked (...,...,Qld,ACCU,29.0,['#407f46'],4,Landfill gas capture,Australia,High Developed,32-126,78
159,159,75,carbon,accu,ACCU,ACCU,6,HIR,Paroo River North Environmental Project,\t\nThis project establishes permanent native ...,...,Qld,ACCU,29.0,"['#dda73a', '#4ca146', '#c5192d', '#407f46', '...",6,HIR,Australia,High Developed,7-35,21


In [20]:

# Define a function to parse the 'un_sd_goals' column and create binary columns for each UN SDG
def parse_un_sd_goals(row):
    if pd.isna(row):
        return []
    else:
        # Remove brackets and split by comma to get the list of SDGs
        return [int(sdg.strip()) for sdg in row.strip('[]').split(',')]

# Apply the function to the 'un_sd_goals' column
data['un_sd_goals_parsed'] = data['un_sd_goals'].apply(parse_un_sd_goals)

# Initialize columns for each UN SDG (1-17)
for sdg in range(1, 18):
    data[f'SDG_{sdg}'] = data['un_sd_goals_parsed'].apply(lambda sdgs: 1 if sdg in sdgs else 0)

# Drop the original 'un_sd_goals' column and the temporary 'un_sd_goals_parsed' column
data.drop(columns=['un_sd_goals', 'un_sd_goals_parsed'], inplace=True)

# Display the first few rows to verify the transformation
data.iloc[:, -18:].head()


,Estimated Job Count,SDG_1,SDG_2,SDG_3,SDG_4,SDG_5,SDG_6,SDG_7,SDG_8,SDG_9,SDG_10,SDG_11,SDG_12,SDG_13,SDG_14,SDG_15,SDG_16,SDG_17
0,1428,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0
1,1428,0,0,0,1,0,0,1,1,0,0,0,0,1,0,1,0,0
2,850,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0
3,714,0,0,0,0,0,1,1,1,0,0,0,0,1,0,0,0,0
4,8,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0


In [21]:
sdg_cols = ['SDG_1','SDG_2','SDG_3','SDG_4','SDG_5','SDG_6','SDG_7','SDG_8','SDG_9','SDG_10','SDG_11','SDG_12','SDG_13','SDG_14','SDG_15','SDG_16','SDG_17']

In [22]:
def is_number(s):
    try:
        float(s)  # Attempt to convert the string to a float
        return True
    except ValueError:
        return


In [23]:
# Ensure the 'Annual Emission Reduction' column is treated as a string
data['Annual Emission Reduction'] = data['Annual Emission Reduction'].astype(str)

# # Apply the function to identify non-numeric entries
# data['is_numeric'] = data['Annual Emission Reduction'].apply(is_number())

# # Filter out non-numeric entries
# data = data[data['is_numeric']]

# Convert 'Annual Emission Reduction' to numeric now that non-numeric entries are removed
data['Annual Emission Reduction'] = data['Annual Emission Reduction'].str.replace(',', '').astype(float)

# Convert 'Estimated Job Count' to numeric if needed
data['Estimated Job Count'] = pd.to_numeric(data['Estimated Job Count'], errors='coerce')

# Drop any rows where 'Estimated Job Count' could not be converted to numeric
data.dropna(subset=['Estimated Job Count'], inplace=True)

# Display the updated info of the dataset
data_info_updated = data.info()
data_info_updated


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 161 entries, 0 to 160
Data columns (total 58 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Unnamed: 0                     161 non-null    int64  
 1   id                             161 non-null    int64  
 2   product_class                  161 non-null    object 
 3   product_type                   161 non-null    object 
 4   product_type_name              161 non-null    object 
 5   product_type_long_name         161 non-null    object 
 6   certificate_project_type       161 non-null    int64  
 7   certificate_project_type_name  156 non-null    object 
 8   name                           161 non-null    object 
 9   description                    154 non-null    object 
 10  region                         157 non-null    float64
 11  region_name                    157 non-null    object 
 12  region.1                       157 non-null    obj

In [24]:
# Calculate SDG Alignment Score (SDGA)
data['SDGA'] = data[sdg_cols].sum(axis=1) 

# Assuming 'Annual Emission Reduction' is DCR and is in the correct numeric format
# Normalize DCR by dividing by the maximum DCR value in the dataset
data['DCR_norm'] = data['Annual Emission Reduction'] 
# Assuming 'Estimated Job Count' is CI
# Normalize CI by dividing by the maximum CI value in the dataset
data['CI_norm'] = data['Estimated Job Count']/ data['Estimated Job Count'].max()


# Assuming a constant ME across all projects, normalized to 1
# Assuming a constant ME across all projects, normalized to 1
data['ME'] = 0
data.loc[data['SDGA'] > 1, 'ME'] = 1.5
data.loc[data['SDGA'] == 1, 'ME'] = 1
  # Or this can be a column based on additional data

# Calculate normalized Social Impact score
data['Social_Impact'] = ((1 - 1 / data['DCR_norm']) * data['SDGA'] * data['CI_norm'] * data['ME']) / 1  # Normalized by 1 since ME is assumed to be constant

# Now data['Social_Impact'] contains the normalized Social Impact score for each project


/var/folders/jh/v1fzjz6d7734l29_2zz94rvr0000gn/T/ipykernel_23830/4136470459.py:15: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '1.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  data.loc[data['SDGA'] > 1, 'ME'] = 1.5


Reasoning Behind 1. Dataset Completeness (Adding Proxy Variables):

In the context of social impact quantification, one of the primary challenges is the absence of direct, measurable indicators for some qualitative outcomes, such as community upliftment, local economic improvement, or well-being. These qualitative outcomes often have no clear numerical representation in datasets that focus on quantitative metrics such as carbon emission reduction or project size. To bridge this gap, proxy variables are introduced to estimate the social impacts indirectly.

Key Reasoning for Adding Proxy Variables:

	1.	Lack of Direct Measurement:
Social impact dimensions, such as improvements in health, job creation, and community engagement, are crucial for understanding the full effects of carbon credit projects. However, many datasets focus solely on environmental impacts like CO2 reduction, leaving these social dimensions unmeasured. By introducing proxy variables (e.g., job creation, project scale), we can estimate the social outcomes in a meaningful way.
	2.	Capturing Multi-Dimensional Impacts:
Carbon credit projects do not only contribute to environmental benefits; they also affect the local communities in terms of employment and socio-economic improvements. These proxy variables allow us to incorporate multiple dimensions of impact into the model, reflecting a more comprehensive understanding of the project’s contribution toward UN SDGs.
	3.	Enriching the Dataset for Model Accuracy:
By enriching the dataset with proxy variables, we allow machine learning models to capture non-linear relationships between quantitative and qualitative variables. For instance, job creation can serve as a surrogate measure for community economic upliftment, which in turn allows us to estimate the wider social benefits of projects.
	4.	Region-Specific Adjustments:
The social impact of a project varies significantly depending on the region where it operates. Projects in developing countries may have a higher social impact due to lower employment baselines or greater reliance on manual labor. The region-specific adjustment (using a country factor) enables the dataset to reflect these regional disparities, making it more accurate in modeling and predicting the real-world outcomes.

Use of Proxy Variables in Statistical Modeling for Social Impact Quantification:

Proxy variables play a critical role in statistical modeling as they provide a quantitative approximation of otherwise intangible social outcomes. Here’s how they are applied in the modeling process:

	1.	Feature Selection and Engineering:
During the data preparation stage, these proxy variables are treated as key features for the model. For instance, job creation and community upliftment become independent variables, while the social impact score (calculated through a model) becomes the dependent variable. These features feed into machine learning models to predict social impact accurately.
	2.	Predictive Modeling:
Machine learning models like Random Forest and Gradient Boosting rely on feature importance to weigh the contribution of each variable. By incorporating proxy variables like project scale and job creation, the models can capture how these proxies influence social outcomes. For instance, regions with high job creation due to large-scale projects will likely show a higher predicted social impact, which is aligned with the aim of quantifying social benefits.
	3.	Interpreting Model Output:
Proxy variables help in interpreting the predictions made by the model. After training the model, we can assess how much weight these variables (e.g., job creation, project size, region-specific factors) carry in determining the overall social impact score. For example, Random Forest and Ridge Regression models will highlight feature importance, showing how much community upliftment (as a proxy) contributes to the predicted social impact.
	4.	Validation and Comparison:
Once the model has predicted the social impact using proxy variables, the results can be compared to real-world data (if available) or validated against expert knowledge to ensure the model’s robustness. The ability of proxy variables to approximate real social outcomes helps validate the overall reliability of the model.

Example of Application:

In the project, we can calculate a social development score for each project using proxy variables such as job creation, project size, and region-specific factors. These proxy variables will then be used as inputs for the machine learning models (OLS, Ridge, Random Forest) to predict the overall social impact score.

For instance:

	•	A large-scale renewable energy project in a developing country may contribute significantly to job creation, which acts as a proxy for community development.
	•	A small-scale project in a developed country might show lower job creation but higher technical expertise, which affects the social impact differently.



In [25]:
data.to_csv('/Users/drishtant/Documents/Masters/CA/core market data/Project_details_1.3.csv')

In [26]:
data

,Unnamed: 0,id,product_class,product_type,product_type_name,product_type_long_name,certificate_project_type,certificate_project_type_name,name,description,...,SDG_13,SDG_14,SDG_15,SDG_16,SDG_17,SDGA,DCR_norm,CI_norm,ME,Social_Impact
0,0,40,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,Grid Connected Wind Power Project in Maharashtra,M/s Bhilwara Green Energy limited (BGEL) is th...,...,1,0,0,0,0,3,73789.0,0.923077,1.5,4.153790
1,1,47,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,300 MW Wind Energy Project,The purpose of the project activity is to gene...,...,1,0,1,0,0,5,603168.0,0.923077,1.5,6.923065
2,2,87,carbon,gs,GS (VER),Gold Standard (VER),31,Energy efficiency,Promoting Improved Cooking Practices in Nigeri...,The project involves manufacturing and distrib...,...,1,0,0,0,0,3,604520.0,0.549451,1.5,2.472523
3,3,346,carbon,gs,GS (VER),Gold Standard (VER),18,Renewables,Kartaldagi Wind Power Plant,Efil Enerji √úretim Ticaret ve Sanayi A.S. (Ef...,...,1,0,0,0,0,4,86046.0,0.461538,1.5,2.769199
4,4,430,carbon,gs,GS (VER),Gold Standard (VER),31,Energy efficiency,"VPA 175 ECOZOOM IMPROVED STOVE PROGRAMME, UGANDA",The VPA involves the distribution of fuel-effi...,...,1,0,0,0,0,3,9998.0,0.005171,1.5,0.023269
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
156,156,301,carbon,accu,ACCU,ACCU,7,Avoided deforestation,Osterley Downs Native Forest Protection Project,\t\nThis project protects the native forest fr...,...,1,0,0,0,0,1,96552.0,0.422754,1.0,0.422749
157,157,82,carbon,accu,ACCU,ACCU,6,HIR,Ashwood Native Forest Protection Project,This project protects the native forest from b...,...,1,0,0,0,0,1,125086.0,0.384615,1.0,0.384612
158,158,81,carbon,accu,ACCU,ACCU,4,Landfill gas capture,Swanbank Landfill Gas Project,\t\nThis project transitioned from a revoked (...,...,1,0,0,0,0,1,77156.0,0.050420,1.0,0.050420
159,159,75,carbon,accu,ACCU,ACCU,6,HIR,Paroo River North Environmental Project,\t\nThis project establishes permanent native ...,...,1,1,1,0,0,6,60482.0,0.013575,1.5,0.122170


Best Parameters: {'model__learning_rate': 0.1, 'model__n_estimators': 100}
Best MSE (Grid Search): 1.9872071484647194
Test MSE: 0.3262431490606871
Test MAE: 0.14065088186356212
Test R²: 0.9609089851506312


RF Best Parameters: {'model__max_depth': None, 'model__n_estimators': 100}
RF Best MSE (Grid Search): 2.0147962210707706
RF Test MSE: 0.15436154464020668
RF Test MAE: 0.11630631198673766
RF Test R²: 0.981504134413013


Silhouette Score for K-Means clustering: 0.2860384191020717


/Users/drishtant/Applications/anaconda3/lib/python3.11/site-packages/sklearn/cluster/_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


ValueError: 
All the 540 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
108 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/drishtant/Applications/anaconda3/lib/python3.11/site-packages/sklearn/model_selection/_validation.py", line 686, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/drishtant/Applications/anaconda3/lib/python3.11/site-packages/sklearn/ensemble/_forest.py", line 345, in fit
    X, y = self._validate_data(
           ^^^^^^^^^^^^^^^^^^^^
  File "/Users/drishtant/Applications/anaconda3/lib/python3.11/site-packages/sklearn/base.py", line 584, in _validate_data
    X, y = check_X_y(X, y, **check_params)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/drishtant/Applications/anaconda3/lib/

,DCR_norm,CI_norm,ME,SDG_count,Country Status,project_method_type_name,region
115,293347.0,0.604396,1.0,1.0,Developed,IFM,48.0
2,604520.0,0.549451,1.5,3.0,Unknown,Energy efficiency,Unknown
123,41204.0,0.923077,1.0,1.0,Developing,Renewables,74.0
45,85680.0,0.032321,1.0,1.0,Developing,Renewables,38.0
42,11233.0,0.032321,1.0,1.0,Developing,Renewables,38.0
...,...,...,...,...,...,...,...
71,23859.0,0.032321,1.5,5.0,Developing,Renewables,38.0
106,1553021.0,0.923077,1.0,1.0,Developing,Renewables,38.0
14,21404.0,0.027149,1.5,3.0,Developing,Energy efficiency,74.0
92,115817.0,0.032321,1.0,1.0,Developing,Renewables,38.0
